In [1]:
#Same script but normalized:

from pathlib import Path
import re
import numpy as np
import pandas as pd
import tifffile
import imageio.v2 as imageio

# Try to import SciPy helpers (for connected components & convex hull)
try:
    from scipy.ndimage import label as cc_label
    from scipy.spatial import ConvexHull
    from scipy.spatial.distance import pdist
    HAVE_SCIPY = True
except Exception:
    cc_label = None
    ConvexHull = None
    pdist = None
    HAVE_SCIPY = False
    print("⚠️ SciPy not fully available. Width will fall back to bounding-box diagonal; "
          "connected-component warnings may be skipped.")

# ============================================================
# Multi-embryo, multi-position volumes + height/width/aspect
# from Cellpose masks, WITH per-embryo normalization.
#
# Directory layout:
#   ROOT_DIR/
#       Embryo 1/
#           PositionA/
#               g1 cell 1/
#                   nuclei/   or nucleus/
#                   membrane/
#               g1 cell 2/
#               s phase cell 1/
#               ...
#           PositionB/
#               ...
#       Embryo 2/
#           ...
#
# For EACH position folder, this script writes:
#   PositionX / "FrustumVolumes_Membrane_Nucleus_byCell.xlsx"
#
# Calibration:
#   Pixel width  = PIXEL_SIZE_UM µm
#   Pixel height = PIXEL_SIZE_UM µm
#   Voxel depth  = Z_STEP_UM µm
#
# For each cell, computes for Membrane and Nucleus:
#   • Frustum volume (µm³)
#   • Sum volume = Σ(Area_z) * Z_STEP_UM  (µm³)
#   • Height (µm) = # of Z slices with mask * Z_STEP_UM
#   • Width (µm)  = largest in-plane diameter across any Z slice
#   • Aspect ratio H/W
#   • # of Z slices used
#
# Additionally, for each embryo (across all positions) it computes
# the median of:
#   - Membrane_Volume_um3
#   - Membrane_Volume_Sum_um3
#   - Membrane_Height_um
#   - Membrane_Aspect_H_over_W
#   - Nucleus_Volume_um3
#   - Nucleus_Volume_Sum_um3
#   - Nucleus_Height_um
#   - Nucleus_Aspect_H_over_W
#
# and adds normalized columns per cell:
#   <col>_NormByEmbryoMedian = <col> / embryo_median(<col>)
# ============================================================

# ------------- USER INPUT -----------------------------------
ROOT_DIR = Path(
    r"C:\Users\brand\OneDrive\Desktop\github code ideas\Fixed High Resolution Membrane and Volume Example\Data"
)  # <-- folder that contains "Embryo 1", "Embryo 2", etc.

# Calibration (update if different for this dataset)
PIXEL_SIZE_UM  = 0.1726335           # µm per pixel (XY)
Z_STEP_UM      = 0.3000000           # µm between Z slices
# ------------------------------------------------------------


# ---------------- helpers -----------------------------------

def _natural_key(s: str):
    """Sort helper: mixed digits/text (for human-like ordering)."""
    parts = re.findall(r'\d+|\D+', str(s))
    return [int(p) if p.isdigit() else p.lower() for p in parts]


# Z index in file names (e.g. "_Z001_", "-Z015")
_RX_ZIDX = re.compile(r'[_\-]Z(\d+)', re.IGNORECASE)


def _is_allowed_mask(p: Path) -> bool:
    """Allowed mask files: cp_masks images or *_seg.npy."""
    suf = p.suffix.lower()
    stem = p.stem.lower()
    if stem.endswith("_cp_masks") and suf in (".png", ".tif", ".tiff"):
        return True
    if stem.endswith("_seg") and suf == ".npy":
        return True
    return False


def _z_index(p: Path) -> int | None:
    """Extract Z index from filename; returns int or None."""
    m = _RX_ZIDX.search(p.stem)
    return int(m.group(1)) if m else None


def _read_mask_bool(path: Path) -> np.ndarray:
    """
    Return boolean 2D mask (True = cell).
    Supports cp_masks images and *_seg.npy (Cellpose formats).
    Keeps largest connected component if multiple (when SciPy is available).
    """
    suf = path.suffix.lower()
    stem = path.stem.lower()

    # ----- load raw mask -----
    if stem.endswith("_cp_masks") and suf in (".png", ".tif", ".tiff"):
        if suf in (".tif", ".tiff"):
            arr = tifffile.imread(str(path))
        else:
            arr = imageio.imread(path)
        if arr.ndim == 3:
            # collapse RGB
            arr = np.mean(arr, axis=2)
        mask = (arr > 0)

    elif stem.endswith("_seg") and suf == ".npy":
        try:
            arr = np.load(path, allow_pickle=False)
        except Exception:
            arr = np.load(path, allow_pickle=True)

        # Unwrap possible dicts / nested structures
        if isinstance(arr, dict):
            if "masks" in arr:
                arr = arr["masks"]
            else:
                for k in ("labels", "label", "mask", "arr_0"):
                    if k in arr:
                        arr = arr[k]
                        break

        if isinstance(arr, np.ndarray) and arr.dtype == object and arr.size == 1:
            v = arr.item()
            if isinstance(v, dict) and "masks" in v:
                arr = v["masks"]
            else:
                arr = v

        if isinstance(arr, np.ndarray) and arr.ndim > 2:
            # If it's a 3D array, you might have [Z,Y,X]; here we assume
            # this loader is called per-slice, so squeeze to 2D if needed.
            arr = arr.squeeze()

        if not isinstance(arr, np.ndarray):
            raise ValueError(f"Unsupported *_seg.npy structure in {path.name}")

        mask = arr if arr.dtype == bool else (arr.astype(float) > 0)

    else:
        raise ValueError(f"Unsupported mask file: {path.name}")

    mask = mask.astype(bool)

    # ----- connected components & multi-cell diagnostics -----
    if cc_label is not None:
        lab, n = cc_label(mask)
        if n >= 2:
            sizes = np.bincount(lab.ravel())
            sizes[0] = 0  # background
            comp_sizes = sizes[1:].tolist()
            z_idx = _z_index(path)
            print(
                f"⚠️ Multi-component mask in file: {path.name} "
                f"(Z={z_idx if z_idx is not None else 'NA'}, components={n}, sizes={comp_sizes})"
            )
            # Keep only the largest component for downstream volume/shape calc
            largest_label = sizes.argmax()
            mask = (lab == largest_label)
    # if no SciPy, we just use the full mask
    return mask


def _collect_masks_zordered(folder: Path):
    """
    From a folder, collect ONE mask per Z slice in Z order.
    Priority per Z:
        1) *_cp_masks.(png|tif|tiff)
        2) *_seg.npy
    Returns (z_indices, masks_list).
    """
    if not folder or not folder.exists():
        return [], []

    files = [p for p in folder.iterdir() if p.is_file() and _is_allowed_mask(p)]
    if not files:
        return [], []

    def rank(pp: Path) -> int:
        # Prefer cp_masks over seg
        return 0 if pp.stem.lower().endswith("_cp_masks") else 1

    by_idx: dict[int, list[Path]] = {}
    others: list[Path] = []

    for p in files:
        idx = _z_index(p)
        if idx is not None:
            by_idx.setdefault(idx, []).append(p)
        else:
            others.append(p)

    chosen: dict[int, Path] = {}
    if by_idx:
        # Use explicit Z indices
        for idx, plist in by_idx.items():
            chosen[idx] = sorted(plist, key=lambda q: (rank(q), _natural_key(q.name)))[0]
    else:
        # No Z indices found; use natural order 1..k
        for k, p in enumerate(
            sorted(others, key=lambda q: (rank(q), _natural_key(q.name))), start=1
        ):
            chosen[k] = p

    ok_indices, ok_masks = [], []
    for z in sorted(chosen.keys()):
        p = chosen[z]
        try:
            mask = _read_mask_bool(p)
            ok_indices.append(z)
            ok_masks.append(mask)
        except Exception as e:
            print(f"⚠️ Could not read mask {p}: {e}")

    return ok_indices, ok_masks


def _volume_frustum(areas_um2: np.ndarray, dz_um: float) -> float:
    """Frustum volume from per-slice areas (Simpson-like formula)."""
    a = np.asarray(areas_um2, float)
    if a.size < 2:
        return float(a.sum() * dz_um) if a.size == 1 else 0.0
    return float(
        np.sum(
            (dz_um / 3.0)
            * (a[:-1] + np.sqrt(np.clip(a[:-1] * a[1:], 0, None)) + a[1:])
        )
    )


def _max_diameter_px_single_slice(mask_2d: np.ndarray) -> float:
    """
    Approximate the largest in-plane diameter (in pixels) for a 2D mask.

    Preferred (if SciPy available):
      - Take convex hull of foreground pixels
      - Compute pairwise distances among hull vertices
      - Use the maximum distance as the diameter

    Fallback (no SciPy):
      - Use bounding-box diagonal as an approximation.
    """
    ys, xs = np.nonzero(mask_2d)
    if ys.size < 2:
        return 0.0

    coords = np.column_stack((ys, xs))  # (N, 2) as [y, x]

    if HAVE_SCIPY and ConvexHull is not None and pdist is not None:
        try:
            hull = ConvexHull(coords)
            hull_pts = coords[hull.vertices]
        except Exception:
            hull_pts = coords
        if hull_pts.shape[0] < 2:
            return 0.0
        dists = pdist(hull_pts.astype(float))
        return float(dists.max()) if dists.size > 0 else 0.0
    else:
        # Fallback: bounding box diagonal
        y_min, y_max = ys.min(), ys.max()
        x_min, x_max = xs.min(), xs.max()
        height_px = y_max - y_min + 1
        width_px  = x_max - x_min + 1
        return float(np.hypot(height_px, width_px))


def _max_diameter_px_from_stack(mask_stack: np.ndarray) -> float:
    """
    For a 3D mask stack [Z, Y, X], compute the largest diameter
    (in pixels) across all Z slices.
    """
    if mask_stack.ndim != 3:
        mask_stack = np.squeeze(mask_stack)
    if mask_stack.ndim != 3:
        # Treat whatever is left as a single slice
        return _max_diameter_px_single_slice(mask_stack.astype(bool))

    max_d = 0.0
    for z in range(mask_stack.shape[0]):
        slice_mask = mask_stack[z].astype(bool)
        d = _max_diameter_px_single_slice(slice_mask)
        if d > max_d:
            max_d = d
    return max_d


def _is_cell_folder(p: Path) -> bool:
    """
    Decide if directory is a 'cell' folder.
    Examples: "g1 cell 1", "s phase cell 3", etc.
    """
    if not p.is_dir():
        return False
    return re.search(r'\bcell\s*\d+', p.name, flags=re.IGNORECASE) is not None


def _extract_cell_number(name: str) -> int | None:
    """Extract numeric cell ID from folder name ('Cell 5', 'g1 cell10', etc.)."""
    m = re.search(r'\bcell\s*(\d+)', name, flags=re.IGNORECASE)
    return int(m.group(1)) if m else None


def _extract_phase_from_name(name: str) -> str:
    """
    Infer 'Phase' from folder name.
    e.g. 'g1 cell 1' -> 'G1', 's phase cell 3' -> 'S', 'g2 cell 2' -> 'G2'
    Returns the raw folder name if nothing matches.
    """
    n = name.lower()
    if "g1" in n:
        return "G1"
    if "g2" in n:
        return "G2"
    if "s phase" in n or n.startswith("s ") or n.startswith("s_"):
        return "S"
    return name  # fallback


def _compute_for_channel(channel_dir: Path):
    """
    Returns:
      frustum_vol_um3,
      sum_vol_um3,
      height_um,
      width_um,
      aspect_hw,
      num_slices
    """
    if channel_dir is None or not channel_dir.exists():
        return np.nan, np.nan, np.nan, np.nan, np.nan, 0

    z_indices, masks = _collect_masks_zordered(channel_dir)
    if not masks:
        print(f"      ℹ️ No usable masks in {channel_dir}")
        return np.nan, np.nan, np.nan, np.nan, np.nan, 0

    shapes = {m.shape for m in masks}
    if len(shapes) > 1:
        print(f"      ⚠️ Inconsistent slice shapes in {channel_dir}: {shapes}; skipping.")
        return np.nan, np.nan, np.nan, np.nan, np.nan, 0

    # Stack as [Z, Y, X]
    mask_stack = np.stack(masks, axis=0)
    num_slices = mask_stack.shape[0]

    # Per-slice area in pixels
    areas_px = mask_stack.reshape(num_slices, -1).sum(axis=1).astype(float)
    # Convert to µm²
    areas_um2 = areas_px * (PIXEL_SIZE_UM ** 2)

    # Volumes
    frustum_vol_um3 = _volume_frustum(areas_um2, Z_STEP_UM)
    sum_vol_um3     = float(areas_um2.sum() * Z_STEP_UM)

    # Height in µm = number of Z slices * Z_STEP_UM
    height_um = num_slices * Z_STEP_UM

    # Width in µm = largest diameter across any Z slice
    max_d_px = _max_diameter_px_from_stack(mask_stack)
    width_um = max_d_px * PIXEL_SIZE_UM

    # Aspect ratio H/W
    if width_um > 0:
        aspect_hw = height_um / width_um
    else:
        aspect_hw = np.nan

    return frustum_vol_um3, sum_vol_um3, height_um, width_um, aspect_hw, num_slices


# ---------------- main scan & compute -----------------------

if not ROOT_DIR.exists():
    raise FileNotFoundError(f"ROOT_DIR does not exist: {ROOT_DIR}")

embryo_dirs = [d for d in ROOT_DIR.iterdir() if d.is_dir()]
embryo_dirs = sorted(embryo_dirs, key=lambda p: _natural_key(p.name))

if not embryo_dirs:
    print(f"⚠️ No embryo folders found in {ROOT_DIR}")

total_cells_global = 0

for embryo_dir in embryo_dirs:
    print(f"\n==============================")
    print(f"Embryo folder: {embryo_dir.name}")
    print(f"==============================")

    pos_dirs = [d for d in embryo_dir.iterdir() if d.is_dir()]
    pos_dirs = sorted(pos_dirs, key=lambda p: _natural_key(p.name))

    if not pos_dirs:
        print(f"  ⚠️ No position folders inside {embryo_dir}")
        continue

    # Collect ALL rows for this embryo (across all positions)
    embryo_rows = []

    for pos_dir in pos_dirs:
        print(f"\n--- Position folder: {pos_dir.name} ---")

        cell_dirs = [d for d in pos_dir.iterdir() if _is_cell_folder(d)]
        cell_dirs = sorted(cell_dirs, key=lambda p: _natural_key(p.name))

        if not cell_dirs:
            print(f"  ⚠️ No 'cell' folders found in {pos_dir}")
            continue

        print(f"  Found {len(cell_dirs)} cell folders in {pos_dir.name}")

        for cdir in cell_dirs:
            cell_label = cdir.name
            cell_num   = _extract_cell_number(cdir.name)
            phase      = _extract_phase_from_name(cdir.name)

            print(f"    → Processing cell folder: {cell_label}  (Phase={phase})")

            # Find Membrane & Nucleus subfolders (case-insensitive)
            subdirs = {d.name.lower(): d for d in cdir.iterdir() if d.is_dir()}

            mem_dir = None
            nuc_dir = None
            for name_lower, dpath in subdirs.items():
                if "membrane" in name_lower:
                    mem_dir = dpath
                if "nucleus" in name_lower or "nuclei" in name_lower:
                    nuc_dir = dpath

            (mem_vol_frustum, mem_vol_sum, mem_height_um, mem_width_um,
             mem_ar, mem_nslices) = _compute_for_channel(mem_dir)

            (nuc_vol_frustum, nuc_vol_sum, nuc_height_um, nuc_width_um,
             nuc_ar, nuc_nslices) = _compute_for_channel(nuc_dir)

            embryo_rows.append({
                "Embryo": embryo_dir.name,         # e.g. "Embryo 1"
                "Position": pos_dir.name,          # e.g. "L ant"

                "Phase": phase,
                "Cell_Number": cell_num,
                "Cell_Folder": cell_label,

                # Membrane volumes
                "Membrane_Volume_um3": mem_vol_frustum,       # frustum
                "Membrane_Volume_Sum_um3": mem_vol_sum,       # Σ area * dz

                # Membrane geometry
                "Membrane_Height_um": mem_height_um,
                "Membrane_Width_um": mem_width_um,
                "Membrane_Aspect_H_over_W": mem_ar,
                "Membrane_Num_Z_slices": mem_nslices,

                # Nucleus volumes
                "Nucleus_Volume_um3": nuc_vol_frustum,        # frustum
                "Nucleus_Volume_Sum_um3": nuc_vol_sum,        # Σ area * dz

                # Nucleus geometry
                "Nucleus_Height_um": nuc_height_um,
                "Nucleus_Width_um": nuc_width_um,
                "Nucleus_Aspect_H_over_W": nuc_ar,
                "Nucleus_Num_Z_slices": nuc_nslices,
            })

    # If no cells at all for this embryo, skip normalization + writing
    if not embryo_rows:
        print(f"  ⚠️ No cells processed in embryo {embryo_dir.name}; skipping.")
        continue

    # Build dataframe for this embryo
    df_emb = pd.DataFrame(embryo_rows)

    # --------------------------------------------------------
    # Per-embryo normalization: compute medians across ALL
    # positions & cells in this embryo, then add normalized
    # columns = value / embryo_median.
    # --------------------------------------------------------

    # Columns to normalize by embryo median
    norm_cols = [
        "Membrane_Volume_um3",
        "Membrane_Volume_Sum_um3",
        "Membrane_Height_um",
        "Membrane_Aspect_H_over_W",
        "Nucleus_Volume_um3",
        "Nucleus_Volume_Sum_um3",
        "Nucleus_Height_um",
        "Nucleus_Aspect_H_over_W",
    ]

    # Compute medians (ignore NaNs)
    embryo_medians = df_emb[norm_cols].median(skipna=True)

    # Add normalized columns
    for col in norm_cols:
        new_col = f"{col}_NormByEmbryoMedian"
        denom = embryo_medians.get(col, np.nan)
        if pd.isna(denom) or denom == 0:
            df_emb[new_col] = np.nan
            print(f"  ⚠️ Embryo {embryo_dir.name}: median for {col} is NaN or 0; "
                  f"normalized column {new_col} set to NaN.")
        else:
            df_emb[new_col] = df_emb[col] / denom

    # --------------------------------------------------------
    # Now split back by Position and write one Excel per position
    # (including the normalized columns).
    # --------------------------------------------------------
    positions_in_embryo = sorted(df_emb["Position"].unique(), key=_natural_key)

    for pos_name in positions_in_embryo:
        df_pos = df_emb[df_emb["Position"] == pos_name].copy()

        if df_pos.empty:
            print(f"  ⚠️ No cells for position {pos_name} in embryo {embryo_dir.name}; skipping Excel.")
            continue

        df_pos = df_pos.sort_values(
            by=["Phase", "Cell_Number", "Cell_Folder"], ignore_index=True
        )

        pos_dir = embryo_dir / pos_name
        out_xlsx = pos_dir / "FrustumVolumes_Membrane_Nucleus_byCell.xlsx"
        df_pos.to_excel(out_xlsx, index=False)

        print(f"  ✅ Done for position {pos_name} in {embryo_dir.name}.")
        print(f"     Wrote Excel: {out_xlsx}")
        print(f"     Total rows (cells) in this position = {len(df_pos)}")

        total_cells_global += len(df_pos)

print("\n========================================")
print("Finished all embryos / positions.")
print(f"Total cells processed across dataset = {total_cells_global}")
print("========================================")



Embryo folder: Embryo #

--- Position folder: Position X ---
  Found 3 cell folders in Position X
    → Processing cell folder: G1 cell 1  (Phase=G1)
    → Processing cell folder: G2 cell 3  (Phase=G2)
    → Processing cell folder: S cell 1  (Phase=S)
  ✅ Done for position Position X in Embryo #.
     Wrote Excel: C:\Users\brand\OneDrive\Desktop\github code ideas\Fixed High Resolution Membrane and Volume Example\Data\Embryo #\Position X\FrustumVolumes_Membrane_Nucleus_byCell.xlsx
     Total rows (cells) in this position = 3

Finished all embryos / positions.
Total cells processed across dataset = 3
